# Creative Sampling via SMC — Full Algorithm

**Goal.** Sample from the tilted distribution
$$q_\lambda(x) \propto p(x)\,e^{\lambda f(x)}, \qquad f(x) = \mathbb{E}_{x'\sim p}[D_{\mathrm{IEM}}(x,x')]$$
Higher $\lambda$ concentrates mass on points that are *plausible under $p$* yet *far from the data manifold* — the creative region.

---

## Pre-computation (done once, before SMC)

| quantity | how | role |
|---|---|---|
| $\gamma$-grid | log-uniform in $[1/(CS)^2,\,2^{10}]$, $N_\gamma=50$, $C=0.75$, $S=$ data scale | IEM integration domain |
| $D_{\mathrm{IEM}}(x, x')$ | integrate score differences over the $\gamma$-grid with $N_\varepsilon=16$ Brownian paths | pairwise creativity distance |
| $\{x'_r\}_{r=1}^R \sim p$ | auto-$R$ by weighted-$\tau$ rank-stability rule | anchor points for $f$ |
| $f(x) = \frac{1}{R}\sum_r D_{\mathrm{IEM}}(x,x'_r)$ | frozen once at ref selection | reward function (fixed throughout SMC) |
| $\lambda_0 = \dfrac{\mathrm{std}(\log p(x'_r))}{\mathrm{std}(f(x'_r))}$ | computed on the refs | balances the two terms in $\log q$ |
| $\lambda = m \cdot \lambda_0$ | $m \in [2, 4]$ chosen by user | tilt strength |
| $G: z \mapsto x$ | EDM probability-flow ODE, $G(\mathcal{N}(0,I)) \approx p$ | prior-preserving latent generator |

---

## SMC Algorithm

**Inputs:** $N$ particles, $\lambda$, $f$, $G$, ESS target $\rho \in (0,1)$, moves per level $K$, step size $s$.

### Initialization

Sample $z^{(n)} \sim \mathcal{N}(0, I)$ for $n = 1,\ldots,N$, set $x^{(n)} = G(z^{(n)})$.  
Compute $f(x^{(n)})$.  
Set log-weights $\log w^{(n)} = 0$, temperature $\beta = 0$.

### Main loop — repeat while $\beta < 1$

**(a) Adaptive temperature step**

Find the largest $\Delta\beta \in (0, 1{-}\beta]$ such that
$$\mathrm{ESS}\!\left(\{w^{(n)} e^{\Delta\beta\,\lambda\, f(x^{(n)})}\}\right) \;\geq\; \rho N$$
where $\mathrm{ESS}(w) = \bigl(\sum_n w_n\bigr)^2 / \sum_n w_n^2$, and $\rho$ is a hyperparameter.
Solved by bisection. Set $\beta \leftarrow \beta + \Delta\beta$.

**(b) Reweight**

$$\log w^{(n)} \;\leftarrow\; \log w^{(n)} + \Delta\beta \cdot \lambda \cdot f(x^{(n)})$$

Particles now target $\pi_\beta(x) \propto p(x)\,e^{\beta\lambda f(x)}$.

**(c) Resample (systematic)**

Draw $N$ indices by systematic resampling from the normalized weights $\tilde{w}^{(n)} = w^{(n)} / \sum_m w^{(m)}$.  
Replace $(z^{(n)}, x^{(n)})$ with the resampled copies. Reset $\log w^{(n)} = 0$.

**(d) MCMC Rejuvenate — $K$ pCN moves** (each invariant to $\pi_\beta$)

For each move repeat for all particles in parallel:

1. **Propose** in latent space (pCN, prior-preserving):
$$z'^{(n)} = \sqrt{1 - s^2}\;z^{(n)} + s\;\xi^{(n)}, \qquad \xi^{(n)} \sim \mathcal{N}(0, I)$$
$$x'^{(n)} = G(z'^{(n)})$$

2. **Accept** with probability (see derivation below — $\log p$ cancels):
$$\alpha = \min\!\left(1,\; e^{\beta\lambda\bigl(f(x'^{(n)}) - f(x^{(n)})\bigr)}\right)$$

3. **Update** if accepted: $z^{(n)} \leftarrow z'^{(n)}$, $x^{(n)} \leftarrow x'^{(n)}$.

4. **Adapt** step size (Robbins–Monro):
$$\log s \;\leftarrow\; \log s + \eta\,(\bar\alpha - \alpha^*)$$
where $\bar\alpha$ is the batch acceptance rate, $\alpha^* \approx 0.23$ is the target and $\eta \approx 0.1$ is the adaptation rate (hyperparameters).

---

## Output

After the final level ($\beta = 1$), optionally resample once more to equalize weights.  
Particles $\{x^{(n)}\}$ are approximately distributed as $q_\lambda$.

**Rank by unnormalized $\log q$:**
$$\mathrm{score}(x^{(n)}) = \log p(x^{(n)}) + \lambda\, f(x^{(n)})$$

The top particles by score are the most creative outputs.

---

### Why $\log p$ cancels in the pCN acceptance ratio

Work in **latent space**. The push-forward $G_\# \mathcal{N}(0,I) = p$ means the SMC target in data space,
$$\pi_\beta(x) \propto p(x)\,e^{\beta\lambda f(x)},$$
pulls back to a latent-space target that replaces $p(x)$ with the Gaussian prior:
$$\rho_\beta(z) \propto \mathcal{N}(z;\,0,I)\,e^{\beta\lambda f(G(z))}, \qquad G_\#\rho_\beta = \pi_\beta.$$

The MH ratio for a proposal $q(z'^{(n)}\mid z^{(n)})$ targeting $\rho_\beta$ is:
$$\frac{\rho_\beta(z'^{(n)})\,q(z^{(n)}\mid z'^{(n)})}{\rho_\beta(z^{(n)})\,q(z'^{(n)}\mid z^{(n)})}
= \underbrace{\frac{\mathcal{N}(z'^{(n)};\,0,I)\,q(z^{(n)}\mid z'^{(n)})}{\mathcal{N}(z^{(n)};\,0,I)\,q(z'^{(n)}\mid z^{(n)})}}_{=\,1\;\text{(pCN reversibility)}}
\cdot\; e^{\beta\lambda\bigl(f(x'^{(n)})-f(x^{(n)})\bigr)}.$$

The first factor equals 1 because the pCN proposal
$z'^{(n)} = \sqrt{1-s^2}\,z^{(n)} + s\,\xi^{(n)},\;\xi^{(n)}\sim\mathcal{N}(0,I)$
satisfies **detailed balance with respect to $\mathcal{N}(0,I)$**:
$$\mathcal{N}(z^{(n)};\,0,I)\;q(z'^{(n)}\mid z^{(n)}) \;=\; \mathcal{N}(z'^{(n)};\,0,I)\;q(z^{(n)}\mid z'^{(n)}).$$

Therefore:
$$\alpha = \min\!\left(1,\; e^{\beta\lambda\bigl(f(x'^{(n)}) - f(x^{(n)})\bigr)}\right)$$
Only $f$ is ever evaluated — $\log p$ does not appear.

## API sketch

```python
# Pre-computation
S       = torch.sqrt(VARS).mean().item()
GAMMAS  = torch.logspace(log2(1/(0.75*S)**2), 10, 50, base=2)
D       = GlobalIEMDistance(p, GAMMAS, num_eps=16)               # or score_fn= for pixels
REFS    = RandomRefs(p, distance=D, fallback='best').select()    # auto-R
f       = lambda x: expected_distance(D, x, REFS)
lam0    = (p.log_p_X(REFS).std() / f(REFS).std()).item()
lam     = m * lam0
G       = density_generator(p, ...)                              # or edm_generator for pixels

# SMC
res     = smc_sample(None, D, REFS, None, lam=lam, n_particles=N,
                     kernel=PCNKernel(G, d), final_resample=True, seed=0)

# Rank
f_out   = f(res.X)
log_q   = p.log_p_X(res.X) + lam * f_out
top_k   = res.X[log_q.argsort(descending=True)[:k]]
```